# Building End-to-End RAG Pipelines with LCEL (Lecture 28)

LangChain Expression Language (LCEL) provides a declarative syntax for composing production RAG pipelines.
This notebook demonstrates:
- Composing chains using the pipe operator (`|`).
- Passing context through `RunnablePassthrough` and custom formatters.
- Streaming responses token-by-token.
- Structured assertions ensuring generation fidelity.

## 1. Setup & Ingestion
Initialize embeddings, documents, and an in-memory vector store.

In [1]:
import os
from typing import List
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

corpus = [
    Document(page_content="Machine learning algorithms build a mathematical model based on training data.", metadata={"id": 1}),
    Document(page_content="Deep learning models use multi-layer neural networks capable of feature learning.", metadata={"id": 2}),
    Document(page_content="Convolutional neural networks are specifically tailored for computer vision tasks.", metadata={"id": 3}),
    Document(page_content="Transformer self-attention enables parallel training across long textual contexts.", metadata={"id": 4}),
]

vector_store = FAISS.from_documents(corpus, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print("Retriever initialized with FAISS.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_13052\171303051.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retriever initialized with FAISS.


## 2. Formatting & Prompt Template Specification
Format retrieved documents and bind prompt variables.

In [2]:
def format_docs(docs: List[Document]) -> str:
    """Format retrieved documents into a single readable context block."""
    return "\n\n".join(f"[Doc {d.metadata.get('id')}]: {d.page_content}" for d in docs)

template = """You are an expert AI tutor. Answer the question based solely on the context provided.
If the answer cannot be found in the context, say 'I cannot find that in the provided documents.'

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("Prompt and LLM ready.")

Prompt and LLM ready.


## 3. LCEL Chain Composition
Compose the declarative pipeline using standard runnables.

In [3]:
rag_chain_lcel = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL chain constructed:")
print(rag_chain_lcel)

LCEL chain constructed:
first={
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021A3AC92EA0>, search_kwargs={'k': 2})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an expert AI tutor. Answer the question based solely on the context provided.\nIf the answer cannot be found in the context, say 'I cannot find that in the provided documents.'\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"), additional_kwargs={})]), ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 

## 4. Pipeline Invocation & Assertion Verification
Invoke the LCEL chain and verify grounded generation.

In [4]:
query = "What are convolutional neural networks tailored for?"
output = rag_chain_lcel.invoke(query)

print(f"Query: {query}")
print(f"Response: {output.strip()}")

assert "vision" in output.lower() or "image" in output.lower(), "Response must reference computer vision"
print("LCEL invocation assertion passed.")

Query: What are convolutional neural networks tailored for?
Response: Convolutional neural networks are specifically tailored for computer vision tasks.
LCEL invocation assertion passed.


## 5. Token-by-Token Streaming with LCEL
Demonstrate low-latency streaming inference.

In [5]:
streaming_query = "Explain how transformer models differ in training."
print(f"Streaming response for query: '{streaming_query}'\n")

collected_chunks = []
for chunk in rag_chain_lcel.stream(streaming_query):
    collected_chunks.append(chunk)
    print(chunk, end="", flush=True)
print("\n")

full_streamed_text = "".join(collected_chunks)
assert len(full_streamed_text) > 10, "Streamed text must not be empty"
print(f"Successfully streamed {len(collected_chunks)} tokens/chunks.")

Streaming response for query: 'Explain how transformer models differ in training.'



I

 cannot

 find

 that

 in

 the

 provided

 documents

.



Successfully streamed 99 tokens/chunks.
